### Q) when we need to use col and when not to use col ?
##### Easy Trick to Remember
#### If you're doing something with the data inside a column
#### OR
#### is Spark asking: "What is inside the column?"
### Use: col("column_name")
##### Examples: 
#####          1. col("salary") + 1000
#####          2. col("salary") > 50000
#####          3. col("salary").cast("int")
#####          4. col("salary").isNull()
#####          5. col("salary") * 2

#### If you're doing something to the column itself
#### OR 
##### Is Spark asking: "Which column?"
### Use: "column_name"
##### Examples: 
#####           1. drop("salary")
#####           2. withColumnRenamed("salary", "emp_salary")
#####           3. fillna({"salary": 0})
#####           4. dropna(subset=["salary"])

In [1]:
import pyspark
print(pyspark.__version__)

4.1.2


In [2]:
# Import SparkSession class.
# SparkSession is the entry point to work with Spark.
from pyspark.sql import SparkSession


# Create a SparkSession object.
# Think of this as starting the Spark Engine.
spark = (
    SparkSession.builder

    # Tell Spark where to run.
    # local[*] means:
    # local   -> Run on this computer.
    # *       -> Use all available CPU cores.
    .master("local[*]")

    # Give a name to your Spark application.
    # This name appears in Spark UI and logs.
    .appName("Employee Data Cleaning")

    # Create a new SparkSession.
    # If one already exists, reuse it.
    .getOrCreate()
)

In [3]:
# Read employees.csv

df = spark.read.csv(

    # CSV file path
    "employees.csv",

    # First row contains column names
    header=True,

    # Automatically detect data types
    inferSchema=True
)

df.show()

+-----------+-----+------+----------+
|employee_id| name|salary|department|
+-----------+-----+------+----------+
|        101|Shiva| 50000|        IT|
|        101|    S|   555|        IT|
|        102| NULL| 45000|        IT|
|        103|  Ram|  NULL|        HR|
|        104| NULL|  NULL|        HR|
+-----------+-----+------+----------+



In [4]:
# Display column names and data types
df.printSchema()

root
 |-- employee_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- department: string (nullable = true)



In [5]:
# Import col() to access DataFrame columns
from pyspark.sql.functions import col

# Import when() to apply conditional logic
from pyspark.sql.functions import when

# Import count() to count rows
from pyspark.sql.functions import count

#### Q) Why we use [] list?
#### ⭐ Easy Memory Trick : ask yourself:
#### "Is Spark asking for multiple things?" means one or more columns
#### If yes, you'll usually use a list.
##### Examples:
##### 1. df.select(["name", "salary"])
##### 2. df.dropDuplicates(["employee_id"])
##### 3. .dropna(subset=["name", "salary"])

In [6]:
# Count NULL values in every column

df.select(

    [

        # Count only NULL values
        count(

            # Check whether the current column value is NULL
            when(

                # Select current column and check for NULL
                col(column_name).isNull(),

                # If NULL, return the column name
                # count() counts these returned values
                column_name

            )

        )

        # Rename the output column
        .alias(column_name)

        # Repeat for every column
        for column_name in df.columns

    ]

).show()

+-----------+----+------+----------+
|employee_id|name|salary|department|
+-----------+----+------+----------+
|          0|   2|     2|         0|
+-----------+----+------+----------+



In [7]:
df.select(
    [
        count(
            when(col(column_name).isNull(), 1)
        ).alias(f"Null values in : {column_name} ")
        
        for column_name in df.columns
    ]
).show()


+-----------------------------+----------------------+------------------------+----------------------------+
|Null values in : employee_id |Null values in : name |Null values in : salary |Null values in : department |
+-----------------------------+----------------------+------------------------+----------------------------+
|                            0|                     2|                       2|                           0|
+-----------------------------+----------------------+------------------------+----------------------------+



In [8]:
df.select(
    count(col("salary")).alias("Total Salary Records")
    ).show()

+--------------------+
|Total Salary Records|
+--------------------+
|                   3|
+--------------------+



In [9]:
# fill null values with unknown in names column

df = df.fillna(
    # Dictionary:
    # Key   -> Column name
    # Value -> Replacement value
    {
        "name" : "Unknown"
    }
    )

df.show()

+-----------+-------+------+----------+
|employee_id|   name|salary|department|
+-----------+-------+------+----------+
|        101|  Shiva| 50000|        IT|
|        101|      S|   555|        IT|
|        102|Unknown| 45000|        IT|
|        103|    Ram|  NULL|        HR|
|        104|Unknown|  NULL|        HR|
+-----------+-------+------+----------+



In [10]:
# fill null values with Avg in salary column

from pyspark.sql.functions import avg

average_salary = df.select(
    avg("salary")
).first()[0]

print(average_salary)

31851.666666666668


In [11]:

df = df.fillna(
    {
        "salary" : average_salary
    }
)

df.show()

+-----------+-------+------+----------+
|employee_id|   name|salary|department|
+-----------+-------+------+----------+
|        101|  Shiva| 50000|        IT|
|        101|      S|   555|        IT|
|        102|Unknown| 45000|        IT|
|        103|    Ram| 31851|        HR|
|        104|Unknown| 31851|        HR|
+-----------+-------+------+----------+



In [12]:
# What is dropDuplicates()?
# dropDuplicates() removes duplicate rows from a Spark DataFrame and returns a new DataFrame.

df = df.dropDuplicates()

df.show()

+-----------+-------+------+----------+
|employee_id|   name|salary|department|
+-----------+-------+------+----------+
|        103|    Ram| 31851|        HR|
|        101|  Shiva| 50000|        IT|
|        101|      S|   555|        IT|
|        104|Unknown| 31851|        HR|
|        102|Unknown| 45000|        IT|
+-----------+-------+------+----------+



In [13]:
df = df.dropDuplicates(["employee_id"])

df.show()

+-----------+-------+------+----------+
|employee_id|   name|salary|department|
+-----------+-------+------+----------+
|        101|  Shiva| 50000|        IT|
|        102|Unknown| 45000|        IT|
|        103|    Ram| 31851|        HR|
|        104|Unknown| 31851|        HR|
+-----------+-------+------+----------+



In [14]:
# What is withColumnRenamed()?
# withColumnRenamed() is used to rename an existing column in a Spark DataFrame. It changes only the column name and not the data.

# df.withColumnRenamed(
#     "old_column_name",
#     "new_column_name"
# )

# change column names employee_id to emp_id, name to emp_name, salary to emp_salary


df = df.withColumnRenamed(
    "employee_id",
    "emp_id"
)

df.show()

+------+-------+------+----------+
|emp_id|   name|salary|department|
+------+-------+------+----------+
|   101|  Shiva| 50000|        IT|
|   102|Unknown| 45000|        IT|
|   103|    Ram| 31851|        HR|
|   104|Unknown| 31851|        HR|
+------+-------+------+----------+



In [15]:
df = df.withColumnRenamed(
    "name",
    "emp_name"
)

df.show()

+------+--------+------+----------+
|emp_id|emp_name|salary|department|
+------+--------+------+----------+
|   101|   Shiva| 50000|        IT|
|   102| Unknown| 45000|        IT|
|   103|     Ram| 31851|        HR|
|   104| Unknown| 31851|        HR|
+------+--------+------+----------+



In [16]:
df = df.withColumnRenamed(
    "salary",
    "emp_salary"
)

df.show()

+------+--------+----------+----------+
|emp_id|emp_name|emp_salary|department|
+------+--------+----------+----------+
|   101|   Shiva|     50000|        IT|
|   102| Unknown|     45000|        IT|
|   103|     Ram|     31851|        HR|
|   104| Unknown|     31851|        HR|
+------+--------+----------+----------+



In [17]:
df = df.withColumnRenamed(
    "department",
    "emp_dept"
)

df.show()

+------+--------+----------+--------+
|emp_id|emp_name|emp_salary|emp_dept|
+------+--------+----------+--------+
|   101|   Shiva|     50000|      IT|
|   102| Unknown|     45000|      IT|
|   103|     Ram|     31851|      HR|
|   104| Unknown|     31851|      HR|
+------+--------+----------+--------+



In [18]:
df.columns

['emp_id', 'emp_name', 'emp_salary', 'emp_dept']

#### What is withColumn()?
#### withColumn() is used to create a new column or replace an existing column in a PySpark DataFrame.
#### It takes the result of an expression (such as arithmetic, cast(), when(), or upper()) and stores it in the specified column.
##### Example: 
#### df = df.withColumn(
####     "column_name",
####     expression
#### )
#### 1. First argument: the column to create or replace.
#### 2. Second argument: the expression that produces the new values.


In [19]:
# df = df.withColumn(
#     "column_name",
#     col("column_name").cast("new_data_type")
# )


df = df.withColumn(
    "emp_salart",    # mistakelly written emp_salart
    col("emp_salary").cast("double")
)

df.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- emp_name: string (nullable = false)
 |-- emp_salary: integer (nullable = true)
 |-- emp_dept: string (nullable = true)
 |-- emp_salart: double (nullable = true)



In [20]:
df = df.drop("emp_salart")

df.show()

+------+--------+----------+--------+
|emp_id|emp_name|emp_salary|emp_dept|
+------+--------+----------+--------+
|   101|   Shiva|     50000|      IT|
|   102| Unknown|     45000|      IT|
|   103|     Ram|     31851|      HR|
|   104| Unknown|     31851|      HR|
+------+--------+----------+--------+



In [21]:
df = df.withColumn(
    "emp_salary",
    col("emp_salary").cast("double")
)

df.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- emp_name: string (nullable = false)
 |-- emp_salary: double (nullable = true)
 |-- emp_dept: string (nullable = true)



### Q) How to round
##### round(column, number_of_decimal_places)

In [22]:
from pyspark.sql.functions import round

df = df.withColumn(
    "emp_bonus",
    round(col("emp_salary") * 0.15, 2)
)

df.show()

+------+--------+----------+--------+---------+
|emp_id|emp_name|emp_salary|emp_dept|emp_bonus|
+------+--------+----------+--------+---------+
|   101|   Shiva|   50000.0|      IT|   7500.0|
|   102| Unknown|   45000.0|      IT|   6750.0|
|   103|     Ram|   31851.0|      HR|  4777.65|
|   104| Unknown|   31851.0|      HR|  4777.65|
+------+--------+----------+--------+---------+



In [23]:
df = df.withColumn(
    "Total_Salary",
    round(col("emp_salary") + col("emp_bonus"), 1)
)

df.show()

+------+--------+----------+--------+---------+------------+
|emp_id|emp_name|emp_salary|emp_dept|emp_bonus|Total_Salary|
+------+--------+----------+--------+---------+------------+
|   101|   Shiva|   50000.0|      IT|   7500.0|     57500.0|
|   102| Unknown|   45000.0|      IT|   6750.0|     51750.0|
|   103|     Ram|   31851.0|      HR|  4777.65|     36628.7|
|   104| Unknown|   31851.0|      HR|  4777.65|     36628.7|
+------+--------+----------+--------+---------+------------+



In [24]:
# Q) # Filter employees whose salary is greater than 45000
# we can use any ways like df.filter() or df.where()
# df.filter(condition)

df.filter(col("emp_salary") > 45000).show()

+------+--------+----------+--------+---------+------------+
|emp_id|emp_name|emp_salary|emp_dept|emp_bonus|Total_Salary|
+------+--------+----------+--------+---------+------------+
|   101|   Shiva|   50000.0|      IT|   7500.0|     57500.0|
+------+--------+----------+--------+---------+------------+



In [25]:
# Q) # Filter employees whose Total_Salary is less than 50000

df.where(col("Total_Salary") < 50000).show()

+------+--------+----------+--------+---------+------------+
|emp_id|emp_name|emp_salary|emp_dept|emp_bonus|Total_Salary|
+------+--------+----------+--------+---------+------------+
|   103|     Ram|   31851.0|      HR|  4777.65|     36628.7|
|   104| Unknown|   31851.0|      HR|  4777.65|     36628.7|
+------+--------+----------+--------+---------+------------+



#### Which order is recommended?
### df.filter(
###     condition
### ).select(
###     required_columns
### )

#### Filter rows ➜ Select required columns ➜ Process the result.

In [27]:
# Q) Filters names where name is not equals to unknown
# display only name and salary column
df.filter(
    col("emp_name") != "Unknown"
    ).select(
        "emp_name",
        "emp_salary"
    ).show()

+--------+----------+
|emp_name|emp_salary|
+--------+----------+
|   Shiva|   50000.0|
|     Ram|   31851.0|
+--------+----------+



#### What is Sorting?
#### Sorting is the process of arranging data in a specific order, either ascending or descending.
#### In PySpark, you can sort a DataFrame using the orderBy() or sort()
##### Example:
#### df.orderBy(col("salary").desc()).show()
#### Q) How to sort by multiple columns?
#### df.orderBy(col("salary").desc(), col("name").asc()).show()

In [28]:
# Sort the data frame based on salary in descending order
df = df.orderBy(
    col("emp_salary").desc()
)

df.show()

+------+--------+----------+--------+---------+------------+
|emp_id|emp_name|emp_salary|emp_dept|emp_bonus|Total_Salary|
+------+--------+----------+--------+---------+------------+
|   101|   Shiva|   50000.0|      IT|   7500.0|     57500.0|
|   102| Unknown|   45000.0|      IT|   6750.0|     51750.0|
|   103|     Ram|   31851.0|      HR|  4777.65|     36628.7|
|   104| Unknown|   31851.0|      HR|  4777.65|     36628.7|
+------+--------+----------+--------+---------+------------+



In [29]:
# Sort the data frame based on salary in descending order
# if salary are same sort those with names in ascending order

df.orderBy(
    col("emp_salary").desc(),
    col("emp_name").asc()
).select(
    "emp_name",
    "emp_salary"
).show()

+--------+----------+
|emp_name|emp_salary|
+--------+----------+
|   Shiva|   50000.0|
| Unknown|   45000.0|
|     Ram|   31851.0|
| Unknown|   31851.0|
+--------+----------+



#### What is groupBy()?
#### Simple Definition
#### groupBy() groups rows that have the same value in one or more columns.

After grouping, we usually perform an aggregation such as:
Count,
Sum,
Average,
Maximum,
Minimum

Important: groupBy() alone is incomplete. It must be followed by an aggregation like count(), sum(), avg(), etc.

In [30]:
# Q) Count how many department are there

df.select("emp_dept").distinct().count()

2

In [31]:
# Q) Count the number of employess in each department

df.groupBy("emp_dept").count().show()

+--------+-----+
|emp_dept|count|
+--------+-----+
|      HR|    2|
|      IT|    2|
+--------+-----+



Think of agg() as a container for aggregation expressions.

Inside that container, you can write one or more aggregation functions (count, sum, avg, etc.), and because each of those returns a Column expression, you can apply .alias() to them.

So agg() is not for alias. It's for performing aggregations. The ability to use alias() is simply one of the advantages of using it.
#### Using groupBy() and agg() together, you can perform complex aggregations on your data while also giving meaningful names to the resulting columns.
#### using agg() to write multiple aggregations in a single groupBy() operation, you can efficiently summarize your data and gain insights from it.

In [32]:
# if we want to display alias name use agg
# here if we use select we can directly write alias agg is not useful
df.groupBy("emp_dept").agg(
    count("*").alias("Employees_Under_Each_department")
).show()

+--------+-------------------------------+
|emp_dept|Employees_Under_Each_department|
+--------+-------------------------------+
|      HR|                              2|
|      IT|                              2|
+--------+-------------------------------+



In [33]:
# Q) Find out the total salaries for each department
df.groupBy(
    "emp_dept"
    ).sum(
        "emp_salary"
    ).show()

+--------+---------------+
|emp_dept|sum(emp_salary)|
+--------+---------------+
|      HR|        63702.0|
|      IT|        95000.0|
+--------+---------------+



In [34]:
# display alias name
from pyspark.sql.functions import sum

df.groupBy("emp_dept").agg(
    sum("emp_salary").alias("Department_salary")
).show()

+--------+-----------------+
|emp_dept|Department_salary|
+--------+-----------------+
|      HR|          63702.0|
|      IT|          95000.0|
+--------+-----------------+



In [35]:
# Q) What is the average salary in each department?

df.groupBy(
    "emp_dept"
).avg(
    "emp_salary"
).show()

+--------+---------------+
|emp_dept|avg(emp_salary)|
+--------+---------------+
|      HR|        31851.0|
|      IT|        47500.0|
+--------+---------------+



In [151]:
# Average salary for each department with alias name

df.groupBy("emp_dept").agg(
    avg("emp_salary").alias("Average_dept_salary")
).show()

+--------+-------------------+
|emp_dept|Average_dept_salary|
+--------+-------------------+
|      HR|            31851.0|
|      IT|            47500.0|
+--------+-------------------+



In [152]:
# Q) For which Department paying more salary
from pyspark.sql.functions import max
from pyspark.sql.functions import min

df.groupBy(
    "emp_dept"
).max(
    "emp_salary"
).show()

+--------+---------------+
|emp_dept|max(emp_salary)|
+--------+---------------+
|      HR|        31851.0|
|      IT|        50000.0|
+--------+---------------+



In [154]:
# Q) For which Department paying minimum salary with alias name

df.groupBy("emp_dept").agg(
    min("emp_salary").alias("Minimum_Dept_Salary")
).show()

+--------+-------------------+
|emp_dept|Minimum_Dept_Salary|
+--------+-------------------+
|      HR|            31851.0|
|      IT|            45000.0|
+--------+-------------------+



#### here we use alias() to select the name of the column in the output DataFrame. It allows us to give a more meaningful name to the aggregated result, making it easier to understand and work with.
#### here we cant use agg() why because we dont use groupBy() or Multiple aggregations. We are just selecting a single column and giving it a new name using alias(). 

In [156]:
df.select(
    round("emp_salary", 2).alias("rounded_salary")
).show()

+--------------+
|rounded_salary|
+--------------+
|       50000.0|
|       45000.0|
|       31851.0|
|       31851.0|
+--------------+



#### Here we use agg() because we use groupBy() and also we use multiple aggregations. We are performing multiple aggregations on the grouped data and giving meaningful names to the resulting columns using alias().

In [157]:
# Q) For each department, show: Total Employees, Total Salary, Average Salary

from pyspark.sql.functions import count, sum, avg

df.groupBy("emp_dept").agg(
    count("*").alias("Total_Employees"),
    sum("emp_salary").alias("Total_Salary"),
    avg("emp_salary").alias("Average_Salary")
).show()

+--------+---------------+------------+--------------+
|emp_dept|Total_Employees|Total_Salary|Average_Salary|
+--------+---------------+------------+--------------+
|      HR|              2|     63702.0|       31851.0|
|      IT|              2|     95000.0|       47500.0|
+--------+---------------+------------+--------------+



#### What is an Outlier?
Simple Definition
An outlier is a value that is much larger or much smaller than the other values in the dataset.

#### What is IQR?
IQR (Inter Quartile Range) is the difference between the third quartile (Q3) and the first quartile (Q1).
Formula:
IQR = Q3 - Q1
It is used to detect outliers.

df.approxQuantile(
    column,
    probabilities,
    relativeError
)

In [36]:
q1, q3 = df.approxQuantile(
    "emp_salary",
    [0.25, 0.75],
    0
)

print(f"Q1: {q1}, Q3: {q3}")

Q1: 31851.0, Q3: 45000.0


Check:
Any NULL values left?
Correct data types?
Salary looks reasonable?
Bonus calculated correctly?

In [37]:
df.show()

+------+--------+----------+--------+---------+------------+
|emp_id|emp_name|emp_salary|emp_dept|emp_bonus|Total_Salary|
+------+--------+----------+--------+---------+------------+
|   101|   Shiva|   50000.0|      IT|   7500.0|     57500.0|
|   102| Unknown|   45000.0|      IT|   6750.0|     51750.0|
|   103|     Ram|   31851.0|      HR|  4777.65|     36628.7|
|   104| Unknown|   31851.0|      HR|  4777.65|     36628.7|
+------+--------+----------+--------+---------+------------+



In [38]:
df.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- emp_name: string (nullable = false)
 |-- emp_salary: double (nullable = true)
 |-- emp_dept: string (nullable = true)
 |-- emp_bonus: double (nullable = true)
 |-- Total_Salary: double (nullable = true)



In [40]:
df.describe().show()

+-------+------------------+--------+-----------------+--------+------------------+------------------+
|summary|            emp_id|emp_name|       emp_salary|emp_dept|         emp_bonus|      Total_Salary|
+-------+------------------+--------+-----------------+--------+------------------+------------------+
|  count|                 4|       4|                4|       4|                 4|                 4|
|   mean|             102.5|    NULL|          39675.5|    NULL| 5951.325000000001|45626.850000000006|
| stddev|1.2909944487358056|    NULL|9262.670619211287|    NULL|1389.4005928816932| 10652.04305426898|
|    min|               101|     Ram|          31851.0|      HR|           4777.65|           36628.7|
|    max|               104| Unknown|          50000.0|      IT|            7500.0|           57500.0|
+-------+------------------+--------+-----------------+--------+------------------+------------------+



#### Save the Data
#### Parquet (recommended in industry):

In [1]:
(
    df.write
    .mode("overwrite")
    .parquet("cleaned_employees")
    )

NameError: name 'df' is not defined